In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import bce # type: ignore
import linear # type: ignore
import sigmoid # type: ignore

from common import test_checkgrad_1, test_compare_1 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Sig_BCE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""

    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None):
        super().__init__()

        #
        # [Linear  + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  nn.Linear(in_features, out_features)
        self.sigmoid = nn.Sigmoid()

        if w is not None:
            with tr.no_grad():
                self.linear.weight.copy_(w)

        if b is not None:
            with tr.no_grad():
                self.linear.bias.copy_(b)

    def forward(self, x):
        return self.model(x)

    def model(self, x):
        z = self.linear(x)
        return self.sigmoid(z)

    def loss(self, p, t):
        return nn.BCELoss(reduction='mean')(p, t)

    def predict(self, x):
        with tr.no_grad():
            return (self.model(x) > 0.5).float()

In [ ]:
class Per_Lin_Sig_BCE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch’s autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  linear.Linear(in_features, out_features, w, b)
        self.sigmoid = sigmoid.Sigmoid()

    def model(self, x):
        z = self.linear(x)
        return self.sigmoid(z)

    def loss(self, p, t):
        return bce.BCE(reduction='mean')(p, t)

    def predict(self, x):
        with tr.no_grad():
            return (self.model(x) > 0.5).float()

    def forward(self, x):
        return self.model(x)

$$ \text{Definition} $$

$$ z = x w + b $$

$$ p = \frac{e^z}{1+e^z} $$

$$ L = \text{bce}(p, t) $$

$$ F(x, w, b, t) = L(x, w, b, t) $$

$$ \text{Derivatives} $$

$$ \frac{dz}{dw} = x $$

$$ \frac{dz}{db} = 1 $$

$$ \frac{dp}{dz} = p \odot (1 - p) $$

$$ \frac{dL}{dp} = (p-t) \oslash (p \odot (1-p)) $$

$$ 
\frac{dL}{dz} = 
\frac{dL}{dp} \frac{dp}{dz} = p - t

$$

$$ \text{Frobenius equality} $$

$$ \left \langle \frac{dF}{dw}, dw \right \rangle + \left \langle \frac{dF}{db}, db \right \rangle = \left \langle \frac{dF}{dL}, dL \right \rangle $$

$$
\left \langle \frac{dF}{dL}, dL \right \rangle =
\frac{dF}{dL} \left \langle \frac{dL}{dz}, dz \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{dw} dw + \frac{dz}{db} db \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{dw} dw \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{db} db \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz}, dw \right \rangle + 
\frac{dF}{dL} \left \langle \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}, db \right \rangle =
$$

$$
\left \langle \frac{dF}{dL} \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz}, dw \right \rangle + 
\left \langle \frac{dF}{dL} \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}, db \right \rangle \implies
$$

$$
\begin{dcases}
    \frac{dF}{dw} = \frac{dF}{dL} \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz} \\
    \frac{dF}{db} = \frac{dF}{dL} \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}
\end{dcases}
$$


In [ ]:
class Per_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(x, w, b):
        z = linear.linear(x, w, b)
        return sigmoid.sigmoid(z)

    @staticmethod
    def __loss(p, y):
        return bce.bce(p, y)

    @staticmethod
    def predict(x, w, b):
        with tr.no_grad():
            return (Per_Lin_Sig_BCE_Gradient_Function.__model(x, w, b) >= 0.5).float()

    @staticmethod
    def forward(ctx, x, w, b, t):
        p = Per_Lin_Sig_BCE_Gradient_Function.__model(x, w, b)
        L = Per_Lin_Sig_BCE_Gradient_Function.__loss(p, t)

        ctx.save_for_backward(x, w, t, p)

        return L

    @staticmethod
    def backward(ctx, dF_dL):
        (x, w, t, p) = ctx.saved_tensors

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = w.shape[1] # Features Out
        N = S * FO

        dz_dw = x
        dL_dz = (1 / N) * (p - t)
        dF_dw = dF_dL * dz_dw.t() @ dL_dz
        dF_db = dF_dL * dL_dz.sum(dim=0)

        return (None, dF_dw, dF_db, None)


class Per_Lin_Sig_BCE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `t`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `w` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if w is None:
            self.weight = nn.Parameter(0.01 * tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(w.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(0.01 * tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Sig_BCE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.__model(x, self.weight.t(), self.bias)

    def loss(self, p, t):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.__loss(p, t)

    def predict(self, x):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.predict(x, self.weight.t(), self.bias)

    def forward(self, x, t):
        return Per_Lin_Sig_BCE_Gradient_Function.apply(x, self.weight.t(), self.bias, t)

In [ ]:
import import_ipynb
from common import test_model_sgd_1, repeat # type: ignore

if __name__ == "__main__":
    test_checkgrad_1(Per_Lin_Sig_BCE_Gradient_Function, 1, 1, 1)
    test_checkgrad_1(Per_Lin_Sig_BCE_Gradient_Function, 2, 2, 2)
    test_checkgrad_1(Per_Lin_Sig_BCE_Gradient_Function, 3, 3, 3)

    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 1, 1, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 1, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 20, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 20, 30)

    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 1, 1, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 1, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 20, 1)
    test_compare_1(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 20, 30)